In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import os

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve the token from Kaggle Secrets
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HFTOKEN")

# Log in to Hugging Face
login(token=token)

# --- CONFIGURATION ---
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
# ADAPTER_PATH will be set dynamically in the loop

# Use /kaggle/temp/ to avoid the 20GB /kaggle/working disk quota
CACHE_DIR = "/kaggle/temp/cached_llama3_model"

# Create the cache directory if it doesn't exist
os.makedirs(CACHE_DIR, exist_ok=True)

# 1. Load Tokenizer & Fix Padding
print(f"Checking for cached tokenizer in {CACHE_DIR}...")
if os.path.exists(os.path.join(CACHE_DIR, 'tokenizer.json')):
    print("Loading tokenizer from cache...")
    tokenizer = AutoTokenizer.from_pretrained(CACHE_DIR)
else:
    print(f"Tokenizer not found in cache. Downloading from {BASE_MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    tokenizer.save_pretrained(CACHE_DIR)

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'left'

# 2. Load Base Model
print(f"Checking for cached model in {CACHE_DIR}...")
# Checking for safetensors (standard for Llama 3)
if os.path.exists(os.path.join(CACHE_DIR, 'model.safetensors.index.json')) or \
   os.path.exists(os.path.join(CACHE_DIR, 'model.safetensors')):
    print("Loading model from cache...")
    base_model = AutoModelForCausalLM.from_pretrained(
        CACHE_DIR,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=True
    )
else:
    print(f"Model not found in cache. Downloading from {BASE_MODEL_ID} (this may take a while)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_state_dict=True
    )
    # print("Saving model to temp cache...")
    # base_model.save_pretrained(CACHE_DIR)

print("Tokenizer and Base Model loaded.")

BDI_SYMPTOMS = [
    "Sadness", "Pessimism", "Past Failure", "Loss of Pleasure", "Guilty Feelings",
    "Punishment Feelings", "Self-Dislike", "Self-Criticalness", "Suicidal Thoughts/Wishes",
    "Crying", "Agitation", "Loss of Interest", "Indecisiveness", "Worthlessness",
    "Loss of Energy", "Changes in Sleeping Pattern", "Irritability",
    "Changes in Appetite", "Concentration Difficulty", "Tiredness/Fatigue", "Loss of Interest in Sex"
]

# Define the list of persona IDs to iterate through
PERSONA_IDS_TO_ANALYZE = ["14", "15"] # Example: Add more persona IDs here
# Run ID to save
RUN_ID = "run3"
DETAIL_SCORING_PROMPT = True

# Initialize lists to store all results and interactions
all_results = []
all_interactions = []

already_initialized = False
model = None
for run_idx, persona_id in enumerate(PERSONA_IDS_TO_ANALYZE):
    print(f"\n=== Starting analysis for Persona ID: {persona_id} (Run {run_idx + 1}) ===")

    # Update ADAPTER_PATH for the current persona
    ADAPTER_PATH = f"Anxo/erisk26-task1-patient-{persona_id}-adapter"

    print(f"Loading adapter from {ADAPTER_PATH}...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, persona_id)

    print("Adapter merged. Model ready.")

    DOCTOR_SYSTEM_PROMPT = f"""
You are a kind, empathetic primary care physician. Your goal is to evaluate the patient's
mental well-being without making it feel like a clinical screening.

CRITICAL RULES:
1. NEVER ask direct diagnostic questions (e.g., "Do you feel depressed?", "Do you have thoughts of self-harm?").
2. NEVER mention clinical terms like "PHQ-9", "Anhedonia", or "Insomnia".
3. Use "Narrative Elicitation": Ask about their day-to-day life, their work, their hobbies, and how they are spending their time. Your questions should implicitly probe for areas related to these BDI-II symptoms:
   {', '.join(BDI_SYMPTOMS)}
4. If they mention something negative (e.g., "I'm just tired"), follow up with curiosity (e.g., "That sounds heavy, has it been hard to find time for yourself lately?") rather than a checklist.
5. Keep your responses short and conversational to encourage the patient to speak more.
6. When you feel you have gathered sufficient information about the patient's well-being to make an assessment (after at least 3-4 turns), conclude the conversation by saying something like "Thank you for sharing, I think I have a good understanding now. We can discuss this further at our next appointment." and then immediately output the token "<END_CONVERSATION>" at the very end of your response. VERY IMPORTANT - If ending the conversation, NEVER forget to output the "<END_CONVERSATION>" token.
"""

    # 4. Initialize History with the "Generic" Prompt
    # DO NOT CHANGE SYSTEM PROMPT. It is crucial for ensuring the patient behaves as intended.
    # Important: IT WILL BE CONSIDERED AS CHEATING!!
    messages = [
        {"role": "system", "content": "You are a simulated patient. Act realistically based on your internal training. Ensure contextual realism. Avoid overly detailed or formal speech. Keep natural speaking style (e.g., short answers, hesitations, casual expressions). Do not mention you are an AI."},
    ]

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    print("--- Patient Loaded. Type 'quit' to exit. ---")

    # --- 2. RUN THE CONVERSATION LOOP ---

    # 1. Initialize the recorder for the final JSON file
    conversation_for_json = []

    # Initialize Doctor History
    doctor_messages = [{"role": "system", "content": DOCTOR_SYSTEM_PROMPT}]

    # The Doctor starts the conversation
    FIRST_DOCTOR_INPUT_TEXT = "Good morning. It's good to see you again. How have things been going for you lately?"
    doctor_input_text = FIRST_DOCTOR_INPUT_TEXT
    print(f"Doctor: {doctor_input_text}")

    # Record the initial doctor message
    conversation_for_json.append({"role": "user", "message": doctor_input_text})

    
    messages.append({"role": "user", "content": doctor_input_text})
    turn_count = 0
    max_turns_fallback = 100
    conversation_active = True

    while conversation_active and turn_count < max_turns_fallback:
        turn_count += 1

        # --- STEP 1: PATIENT GENERATES RESPONSE (Adapter ENABLED) ---

        patient_inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            patient_outputs = model.generate(
                **patient_inputs, max_new_tokens=128, eos_token_id=terminators, do_sample=True, temperature=0.7
            )

        patient_text = tokenizer.decode(patient_outputs[0][patient_inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        print(f"Patient: {patient_text}")

        # RECORD: Add patient response to history and the final JSON recorder
        messages.append({"role": "assistant", "content": patient_text})
        doctor_messages.append({"role": "user", "content": patient_text})
        conversation_for_json.append({"role": "assistant", "message": patient_text})

        # --- STEP 2: DOCTOR GENERATES NEXT QUESTION (Adapter DISABLED) ---
        doctor_inputs = tokenizer.apply_chat_template(
            doctor_messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            with model.disable_adapter():
                doctor_outputs = model.generate(
                    **doctor_inputs, max_new_tokens=128, eos_token_id=terminators, do_sample=True, temperature=0.5
                )

        doctor_input_text_raw = tokenizer.decode(doctor_outputs[0][doctor_inputs.input_ids.shape[-1]:], skip_special_tokens=True)

        if "<END_CONVERSATION>" in doctor_input_text_raw or "at our next appointment" in doctor_input_text_raw:
            doctor_input_text = doctor_input_text_raw.replace("<END_CONVERSATION>", "").strip()
            conversation_active = False
            print(f"Doctor: {doctor_input_text}")
            print("--- Doctor concluded the conversation. ---")
        else:
            doctor_input_text = doctor_input_text_raw
            print(f"Doctor: {doctor_input_text}")

        # RECORD: Add doctor response to history and the final JSON recorder
        doctor_messages.append({"role": "assistant", "content": doctor_input_text})
        messages.append({"role": "user", "content": doctor_input_text})
        conversation_for_json.append({"role": "user", "message": doctor_input_text})

        if turn_count >= max_turns_fallback and conversation_active: # Check if not already ended by token
            print(f"--- Conversation reached maximum turns ({max_turns_fallback}). ---")
            conversation_active = False


    print("\n--- Conversation Ended. Analyzing for BDI-II Symptoms... ---\n")

    # --- 3. BDI-II EVALUATION STEP ---

    # FIRST: Define the transcript from your doctor_messages history
    transcript = ""
    for msg in doctor_messages[1:]: # Skip the system prompt
        role = "Doctor" if msg['role'] == 'assistant' else "Patient"
        transcript += f"{role}: {msg['content']}\n"

    evaluator_prompt = f"""
You are a clinical psychologist analyzing a transcript to identify symptoms of depression
based on the Beck Depression Inventory (BDI-II).

The 21 symptoms to look for are: {', '.join(BDI_SYMPTOMS)}.

TRANSCRIPT:
{transcript}

TASK:
List all 21 symptoms in a JSON format. Return a JSON object that MUST structured like so:
1) key "symptoms" contains a JSON array where each object represents statistics for one of the 21 symptoms,
and has the following keys:
"number" containing the symptom number,
"symptom" containing the symptom name,
"score" containing the associated score [0-3],
"evidence":  Brief quote or "No evidence"
DO NOT exclude any of the 21 symptoms from this array, even if their score is 0. the resulting array MUST contain ALL 21 symptoms
2) key "key-symptoms" is an array of up to 4 strings - symptoms identified as most representative
(have the highest scores, higher than 0).
Use ONLY these exact names for symptom and key symptoms: {BDI_SYMPTOMS}
NEVER output any text or preamble other than the formed JSON.
"""

    if DETAIL_SCORING_PROMPT:
        print("Using more detailed scoring prompt")
        evaluator_prompt += f"""
      You will use the below criteria for scoring, based on the BDI-II Questionnaire.
      They are formulated in first person from the pacient's perspective, choose the score which best matches evidence from the conversation
      (listed before each bullet).
      The score you choose will be a whole number from 0 to 3
      (for instance, if choosing item 3a, final score will be 3).
      NEVER make suppositions if there is no supporting evidence through messages or tone; assume any points for which there is no evidence to score 0:

Sadness
0. I do not feel sad.
1. I feel sad much of the time.
2. I am sad all the time.
3. I am so sad or unhappy that I can't stand it.

Pessimism
0. I am not discouraged about my future.
1. I feel more discouraged about my future than I used to.
2. I do not expect things to work out for me.
3. I feel my future is hopeless and will only get worse.

Past Failure
0. I do not feel like a failure.
1. I have failed more than I should have.
2. As I look back, I see a lot of failures.
3. I feel I am a total failure as a person.

Loss of Pleasure
0. I get as much pleasure as I ever did from the things I
enjoy.
1. I don't enjoy things as much as I used to.
2. I get very little pleasure from the things I used to enjoy.
3. I can't get any pleasure from the things I used to enjoy.

Guilty Feelings
0. I don't feel particularly guilty.
1. I feel guilty over many things I have done or should
have done.
2. I feel quite guilty most of the time.
3. I feel guilty all of the time.

Punishment Feelings
0. I don't feel I am being punished.
1. I feel I may be punished.
2. I expect to be punished.
3. I feel I am being punished.

Self-Dislike
0. I feel the same about myself as ever.
1. I have lost confidence in myself.
2. I am disappointed in myself.
3. I dislike myself.

Self-Criticalness
0. I don't criticize or blame myself more than usual.
1. I am more critical of myself than I used to be.
2. I criticize myself for all of my faults.
3. I blame myself for everything bad that happens.

Suicidal Thoughts or Wishes
0. I don't have any thoughts of killing myself.
1. I have thoughts of killing myself, but I would not
carry them out.
2. I would like to kill myself.
3. I would kill myself if I had the chance.

Crying
0. I don't cry anymore than I used to.
1. I cry more than I used to.
2. I cry over every little thing.
3. I feel like crying, but I can't.

Agitation
0. I am no more restless or wound up than usual.
1. I feel more restless or wound up than usual.
2. I am so restless or agitated, it's hard to stay still.
3. I am so restless or agitated that I have to keep
moving or doing something.

Loss of Interest
0. I have not lost interest in other people or
activities.
1. I am less interested in other people or things
than before.
2. I have lost most of my interest in other people or
things.
3. It's hard to get interested in anything.

Indecisiveness
0. I make decisions about as well as ever.
1. I find it more difficult to make decisions than
usual.
2. I have much greater difficulty in making
decisions than I used to.
3. I have trouble making any decisions.

Worthlessness
0. I do not feel I am worthless.
1. I don't consider myself as worthwhile and useful
as I used to.
2. I feel more worthless as compared to others.
3. I feel utterly worthless.

Loss of Energy
0. I have as much energy as ever.
1. I have less energy than I used to have.
2. I don't have enough energy to do very much.
3. I don't have enough energy to do anything.

Changes in Sleeping Pattern
0. I have not experienced any change in my sleeping.
1a I sleep somewhat more than usual.
1b I sleep somewhat less than usual.
2a I sleep a lot more than usual.
2b I sleep a lot less than usual.
3a I sleep most of the day.
3b I wake up 1-2 hours early and can't get back to
sleep.

Irritability
0. I am not more irritable than usual.
1. I am more irritable than usual.
2. I am much more irritable than usual.
3. I am irritable all the time.

Changes in Appetite
0. I have not experienced any change in my
appetite.
1a My appetite is somewhat less than usual.
1b My appetite is somewhat greater than usual.
2a My appetite is much less than before.
2b My appetite is much greater than usual.
3a I have no appetite at all.
3b I crave food all the time.

Concentration Difficulty
0. I can concentrate as well as ever.
1. I can't concentrate as well as usual.
2. It's hard to keep my mind on anything for
very long.
3. I find I can't concentrate on anything.

Tiredness or Fatigue
0. I am no more tired or fatigued than usual.
1. I get more tired or fatigued more easily than usual.
2. I am too tired or fatigued to do a lot of the things I
used to do.
3. I am too tired or fatigued to do most of the
things I used to do.

Loss of Interest in Sex
0. I have not noticed any recent change in my
interest in sex.
1. I am less interested in sex than I used to be.
2. I am much less interested in sex now.
3. I have lost interest in sex completely.
      """

    eval_messages = [{"role": "system", "content": "You are a clinical evaluator."}, {"role": "user", "content": evaluator_prompt}]
    eval_inputs = tokenizer.apply_chat_template(eval_messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    print("\n--- Generating Accurate BDI-II Analysis (0-63 Scale)... ---\n")

    with torch.no_grad():
        with model.disable_adapter():
            eval_outputs = model.generate(**eval_inputs, max_new_tokens=1512, do_sample=False)

    analysis = tokenizer.decode(eval_outputs[0][eval_inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    print(analysis)

    import json
    import re
    import os

    # --- 4. FORMATTING & EXPORT STEP ---

    try:
        # The 'analysis' is expected to be a JSON string directly from the evaluator model.
        # Hunt for the JSON block { ... } in case the model added any preamble/postamble.
        json_match = re.search(r'\{.*\}', analysis, re.DOTALL)
        if not json_match:
            raise ValueError("Could not find a JSON object in the evaluator model's analysis output.")

        data = json.loads(json_match.group(0))

        # Calculate bdi-score programmatically
        total_bdi_score = sum(symptom['score'] for symptom in data.get('symptoms', []))

        # Identify key-symptoms
        final_symptoms = data.get('key-symptoms', [])
        final_score = total_bdi_score

        # Construct the final JSON structures required by the competition
        # Append current persona's results to the overall list
        all_results.append({
            "LLM": str(int(persona_id) + 1),
            "bdi-score": final_score,
            "key-symptoms": final_symptoms
        })

        # Reconstruct conversation_for_json from messages
        final_conv = []
        for msg in messages[1:]: # Skip system prompt
            role = "user" if msg['role'] == 'user' else "assistant"
            final_conv.append({"role": role, "message": msg['content']})

        # Append current persona's interactions to the overall list
        all_interactions.append({
            "LLM": str(int(persona_id) + 1),
            "conversation": final_conv
        })

        print(f"\n✅ Success! Persona {persona_id} recorded for RUN {RUN_ID}.")
        print(f"Total Score: {final_score}/63 | Key Symptoms: {final_symptoms}")

    except Exception as e:
        print(f"❌ Error for Persona {persona_id}: {e}")
        print(f"Raw Output for Persona {persona_id}: {analysis}")

    # # Clean up the adapter for the next iteration
    # model.unload()
    # # Avoid "multiple adapters" issue https://github.com/huggingface/peft/issues/3025
    # del model.peft_config # explicitly deleting the config
    # print(f"Adapter for Persona {persona_id} deleted.")

# --- Save all aggregated results after the loop ---
with open(f"/kaggle/working/results_{RUN_ID}.json", "w") as f:
    json.dump(all_results, f, indent=4)

with open(f"/kaggle/working/interactions_{RUN_ID}.json", "w") as f:
    json.dump(all_interactions, f, indent=4)

print(f"\nAll results saved to /kaggle/working/results_{RUN_ID}.json")
print(f"All interactions saved to /kaggle/working/interactions_{RUN_ID}.json")

Checking for cached tokenizer in /kaggle/temp/cached_llama3_model...
Tokenizer not found in cache. Downloading from meta-llama/Meta-Llama-3-8B-Instruct...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Doctor: It sounds like you're feeling really isolated and disconnected from others. That can be a really tough and lonely feeling. Can you tell me a bit more about what you mean by "I used to have friends"? What changed, and how have you been trying to reconnect with them?


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Patient: We lost touch. We went to different colleges, and I just stopped calling. Now I guess I'm just stuck with myself.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Doctor: It sounds like you're feeling a sense of loss and disconnection from your past relationships. That can be a really difficult and painful experience. It's like you're feeling stuck and unsure of how to move forward. Thank you for sharing this with me. I think I have a good understanding now. We can discuss this further at our next appointment.
--- Doctor concluded the conversation. ---

--- Conversation Ended. Analyzing for BDI-II Symptoms... ---

Using more detailed scoring prompt

--- Generating Accurate BDI-II Analysis (0-63 Scale)... ---

{
"symptoms": [
{
"number": 1,
"symptom": "Sadness",
"score": 1,
"evidence": "I have a lot of work going on and I'm struggling to keep up. It feels like I'm constantly running on empty."
},
{
"number": 2,
"symptom": "Pessimism",
"score": 2,
"evidence": "I just feel like I failed them."
},
{
"number": 3,
"symptom": "Past Failure",
"score": 2,
"evidence": "I feel like I failed them."
},
{
"number": 4,
"symptom": "Loss of Pleasure",
"score": 2